In [ ]:
import numpy as np
import matplotlib.pyplot as plt

**KRAUS OPERATOR**- In real world, no quantum computer or quantum system is perfectly isolated. Environmental noise interacts with qubit, and kraus operators provide a way to mathematically calculate exactly how that environment alters quantum state without needing to track the environment details



**Mathematical Formulation**- The environment can trigger different scenarios. Each scenario has it's own kraus operator. for each scenario we sandwich the quantum state between the operator and its conjugate transpose K(rho)K^dagger. add up all the scenarios together to get final state rho_out = sum_k (K_k(rho)K_k^dagger)


For a kraus operator to work physically they must obey a key rule called completeness relation ->> sum_k(K_k^dagger*K_k)=I(identity matrix)
This is basically law of conservation it guarantees that if you add up the probabilities of all the possible scenarios happening to the qubit the total probability is equal to "1"

In [ ]:
def check_completeness(kraus_ops, atol=1e-8):
  """Parameters
  kraus_ops : list of kraus operators {K_k}
  atol : a little bit of tolerance is given because floating points sum may have a bit error"""
  dim = kraus_ops[0].shape[0]
  "this gives the dimension of operators"
  sum = np.zeros((dim,dim), dtype=complex)
  #a null vector of same dimensions as kraus operator
  for K in kraus_ops:
    K = np.array(K, dtype=complex)
    #to change to complex if real entries are there
    sum += K.conj().T@K
  identity_matrix = np.eye(dim)
  is_valid = np.allclose(sum, identity_matrix, atol=atol)
  #np.allclose checks if two arrays are element wise equal with a specific tolerance
  return is_valid, sum

In [ ]:
def validate_T1_T2(T1,T2):
  if T2> 2*T1 :
    raise ValueError(f"for parameters T1, T2 T2={T2} <=2*T1 = {2*T1} must be true")

now let us define a function to execute kraus operator mathematically with parameters being input density matrix(rho), and a list of kraus operators (kraus_ops)

In [ ]:
def apply_kraus(rho, kraus_ops):
  """Parameters
  rho : input density matrix
  kraus_ops: list of kraus operators {K_k}

  Returns output density matrix rho_out"""
  rho = np.array(rho, dtype=complex)
  #this line is implemented because if user even gives real inputs they need to be considered as complex
  dim = rho.shape[0]
  rho_out= np.zeros((dim,dim), dtype=complex)
  #created a null matrix with same dimensions as input density matrix
  for K in Kraus_ops:
    K = np.array(K, dtype=complex)
    rho_out += K@rho@K.conj().T
    #applied the kraus operator formulae rho_out = sum_k(K_k(rho)K_k^dagger)
  return rho_out

**Amplitude dampinng channel**:
To model energy loss where a qubit decays from its excited state |1> to ground state |0>, we use the amplitude damping channel. this process is exactly described by two kraus operators
k0 = {[1,0],
      [0, sqrt[1-gamma]}

      
k1 = {[0, sqrt(p)],
      [0,  0]}

In [ ]:
def amplitude_damping_channel(gamma):
  k0 = np.array([[1,0],[0,np.sqrt(1-gamma)]])
  k1 = np.array([[0,np.sqrt(gamma)],[0,0]])
  return [k0,k1]

to extract time T1 we connect the parameter gamma(t) to the time elapsed during the quantum operation as a standard curve equation gamma(t) = 1 - e^(-t/T1)

**Phase damping**: used to model and calculate (T2) this is also described by exactly two kraus operators
k0 = {[1, 0],
      [0, sqrt(1-lam)]}

k1 = {[0, 0],
      [0, sqrt(lam)]}

it is a realistic continuous environment driven decay

In [ ]:
def phase_damping_channel(lam):
  """Kraus operator for pure dephasing with lam between 0 and 1"""
  k0 = np.array([[1, 0], [0, np.sqrt(1-lam)]])
  k1 = np.array([[0,0], [0, np.sqrt(lam)]])
  return [k0, k1]

now let us apply both noises simultaneously and define a combined relaxation channel.

In [ ]:
def combined_relaxation_channel(t, T1, T2):
    """
    Kraus operators combining amplitude damping (T1) and extra pure
    dephasing (T_phi) so that the overall coherence decay matches T2.

    Requires T2 <= 2*T1 (physical constraint).
    """
    gamma = 1 - np.exp(-t / T1)

    # Solve 1/T2 = 1/(2*T1) + 1/T_phi for the dephasing rate lambda(t)
    rate_phi = 1.0 / T2 - 1.0 / (2 * T1)
    #check wether the obtained rate is valid or not
    if rate_phi < 0:
        raise ValueError("T2 must be <= 2*T1 for a physical channel.")
    lam = 1 - np.exp(-2 * t * rate_phi) if rate_phi > 0 else 0.0

    # Combine by composing the two channels' Kraus operators
    amp_ops = amplitude_damping_channel(gamma)
    phase_ops = phase_damping_channel(lam)

    combined = []
    for Ka in amp_ops:
        for Kp in phase_ops:
            combined.append(Kp @ Ka)
            """Kp@Ka or Ka@Kp order doesn't matter here because amplitude damping and dephasing are
            commuting quantum channels. Hence, both Kp@Ka and Ka@Kp produce the same overall effect
            on the qubit"""
    return combined

Now let us simulate a qubit through the combined T1/T2 channel at increasing times and record:
1. population in |1>(for T1 fit, starting from |1>)



   2.coherence magnitude |rho_1| (for T2 fit starting from |+>)

In [ ]:
def simulate_relaxation(T1, T2, t_max, n_points=50):
  """Parameters
  T1: energy relaxation time
  T2: phase relaxation time
  t_max: maximum simulation time
  n_points: number of time instants between 0 and t_max.
  returns
  times: array of time points at which the simulation was performed
  excited state population P(|1>) at each time and
  an array of coherence magnitudes |rho_1| at each time"""
  times = np.linspace(0, t_max, n_points)
  #this creates equally spaced time values from 0 to t_max
  one = np.array([[0], [1]])
  #T1 curve: start in excited state |1>
  rho1_0= one@one.conj().T
  #this creates a density matrix, initial state used for measuring T1
  pop1 = []
  #empty list to store probability of finding the qubit in |1> after each time step

  plus = np.array([[1],[1]])/np.sqrt(2)
  #T2 curve: start in |+> to track coherence decay
  rho2_0 = plus@plus.conj().T
  #creates density matrix of |+>
  coherence = []
  for t in times:
    kraus_ops = combined_relaxation_channel(t, T1, T2)
    rho1 = apply_kraus(rho1_0, kraus_ops)
    pop1.append(rho1_t[1,1].real)
    #rho1_t[1,1] is the probability that qubit is still in |1> only real part is stored because probabilities are real numbers
    rho2 = apply_kraus(rho2_0, kraus_ops)
    coherence.append(np.abs(rho2_t[0,1]))
    #give its magnitude, which decreases as the qubit loses coherence
    return times, np.array(pop1), np.array(coherence)


Now let us estimate the values of T1 and T2 using above simulation

In [ ]:
def fit_T1_T2(times, po1, coherence):
  """ T1 from population decay: P1(t) = exp(-t/T1)
  T2 from coherence decay: |rho_1(t)| = 0.5*exp(-t/T2)
  let us use linear regressio on the log of the data"""
  mask1 = pop1 >1e-10
  #to ignore very small values as logarithm value will tend to infinity
  slope1, k1= np.polyfit(times[mask1], np.log(pop1[mask1]),1)
  #as slope1 = -1/T1_fit
  T1_fit = -1/slope1
  mask2 = coherence > 1e-10
  slope2, k2 = np.polyfit(times[mask2], np.log(coherence[mask2]),1)
  #as slope 2 = -1/T2_fit
  T2_fit = -1/slope2
  return T1_fit, T2_fit